In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None, "display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
from src import config as C

# 01 — İş Problemini Anlama (Business Understanding)

**Bu notebook'ta ne yapıyoruz?** Hiç kod yazmadan önce şu soruyu cevaplıyoruz: *Bu model kime, hangi karar için lazım?*
Veri bilimi projelerinin en sık başarısız olma nedeni kötü model değil, **yanlış sorudur**.

## Problem
Bir abonelik şirketi (telekom/dijital hizmet) her ay müşteri kaybediyor. Yeni müşteri kazanmak, mevcut müşteriyi tutmaktan genellikle daha pahalıdır.
Amaç: **ayrılma ihtimali yüksek müşterileri önceden bulup**, sınırlı bütçeyle elde tutma kampanyası yapmak.

## Bu bir ne tür problem?
* Hedef değişken `churn` (0/1) → **ikili sınıflandırma**.
* Çıktı olarak sadece 0/1 değil, **risk skoru (olasılık)** istiyoruz; çünkü kampanya bütçesi kadar müşteriyi riske göre sıralayıp arayacağız.

## Başarıyı nasıl ölçeceğiz?
| Katman | Metrik | Neden |
|---|---|---|
| Model (teknik) | **PR-AUC**, ROC-AUC, Brier | Dengesiz sınıfta sıralama kalitesi + olasılıkların doğruluğu |
| Karar (operasyonel) | Recall, Precision, aranan müşteri yüzdesi | "Kaç ayrılanı, kaç aramayla yakalıyoruz?" |
| İş (finansal) | **Kampanya net değeri (TL)** | Sonuçta para kazandırmayan model işe yaramaz |

> **Accuracy'yi bilerek ana metrik yapmıyoruz.** Aşağıda göreceğin gibi, herkese "kalır" diyen bir model bile ~%92 Accuracy alır.

In [2]:
data_dictionary = pd.read_csv(C.ROOT / "data" / "raw" / "data_dictionary.csv", encoding="utf-8-sig")
display(data_dictionary)

,column,type,description
0,customer_id,string,Müşteri kimliği; modele doğrudan verme
1,age,numeric,Yaş; beklenen aralık 18-100
2,region,categorical,Bölge
3,device_type,categorical,En çok kullanılan cihaz
4,plan,categorical,Basic / Standard / Premium
5,contract_type,categorical,Monthly / Annual / Two-Year
6,tenure_months,numeric,"Abonelik süresi (ay), 1-72"
7,monthly_usage_hours,numeric,Aylık kullanım saati; makul aralık 0-100
8,monthly_fee_tl,numeric,"Aylık ücret (TL), pozitif olmalı"
9,discount_rate,numeric,"İndirim oranı, 0-0.20"


## Sınıf dengesizliği neden önemli? (Küçük bir deney)
Aşağıda hiçbir şey öğrenmeyen bir "model" kuruyoruz: **herkese churn=0 diyor.**

In [3]:
raw = pd.read_csv(C.RAW_PATH, dtype={"monthly_fee_tl": "object"})
churn_rate = raw["churn"].mean()
display(Markdown(f"Ham veride churn oranı: **{churn_rate:.2%}**  →  herkese 'kalır' diyen modelin Accuracy'si: **{1-churn_rate:.2%}**, "
                 f"yakaladığı ayrılan müşteri: **0**."))

Ham veride churn oranı: **8.30%**  →  herkese 'kalır' diyen modelin Accuracy'si: **91.70%**, yakaladığı ayrılan müşteri: **0**.

Yani **%91'in üstünde Accuracy'ye sahip, iş değeri sıfır** bir model mümkün. Bu yüzden Precision, Recall, PR-AUC ve para üzerinden değerlendireceğiz.

## İş varsayımları (kampanya ekonomisi)
Elimizde gerçek kampanya maliyeti yok; bu yüzden **açıkça yazılmış varsayımlar** kullanıyoruz ve sonucun bu varsayımlara ne kadar duyarlı olduğunu da (Notebook 08) göstereceğiz.

In [4]:
display(pd.Series(C.ASSUMPTIONS, name="varsayım").to_frame())
fee = 351.86
value = fee * C.ASSUMPTIONS["customer_value_months"] * C.ASSUMPTIONS["margin"]
gain_if_success = C.ASSUMPTIONS["offer_success_rate"] * value
breakeven = C.ASSUMPTIONS["offer_cost_tl"] / gain_if_success
display(Markdown(f"Ortalama müşteri için: kurtarılan müşterinin değeri ≈ **{value:,.0f} TL**, beklenen kazanç (başarı olasılığıyla) ≈ **{gain_if_success:,.0f} TL**. "
                 f"Aranan müşteri başına maliyet **{C.ASSUMPTIONS['offer_cost_tl']:.0f} TL** ise, kampanyanın **başabaş precision'ı ≈ %{breakeven:.1%}**. "
                 f"Yani aradığımız grubun en az %{breakeven:.0%}'i gerçekten ayrılacaksa kampanya kâr eder. Taban oran %{churn_rate:.1%} olduğu için model bu eşiği aşan bir *kesim* bulabilmeli."))

,varsayım
offer_cost_tl,50.000
offer_success_rate,0.300
customer_value_months,12.000
margin,0.400


Ortalama müşteri için: kurtarılan müşterinin değeri ≈ **1,689 TL**, beklenen kazanç (başarı olasılığıyla) ≈ **507 TL**. Aranan müşteri başına maliyet **50 TL** ise, kampanyanın **başabaş precision'ı ≈ %9.9%**. Yani aradığımız grubun en az %10%'i gerçekten ayrılacaksa kampanya kâr eder. Taban oran %8.3% olduğu için model bu eşiği aşan bir *kesim* bulabilmeli.

## Proje riskleri / sınırlılıklar (baştan yazalım)
1. **Veri sentetik.** Gerçek müşteri davranışını temsil ettiği garanti değil.
2. **Az sayıda pozitif örnek** (~148 ayrılan). Test kümesinde sadece ~30 ayrılan olacak → metrikler oynak; bunu güven aralığıyla göstereceğiz.
3. **Korelasyon ≠ nedensellik.** "Memnuniyeti düşük müşteri ayrılıyor" demek, memnuniyeti artırınca ayrılmayacağı anlamına gelmez. Kampanya etkisi A/B testiyle ölçülmeli.
4. **Eşik kararı iş maliyetine bağlı.** Bizim maliyet rakamlarımız varsayımdır.